# 🔥 ClusterOps: Thermal GPU Balancer — GRPO Training

**Train a lightweight LLM (Llama-3.2-1B) to manage a GPU data center. Optimized for fast iteration on Colab T4.**

---

## 1. Install Dependencies

In [ ]:
# Install dependencies
!pip install unsloth vllm -q
!pip install trl>=0.12.0 transformers>=4.45.0 -q
!pip install fastapi uvicorn requests pydantic openenv-core -q
!pip install matplotlib numpy -q
print('✅ Dependencies installed')

## 2. Clone & Start ClusterOps Environment Server

In [ ]:
import subprocess, time, requests, os, sys

REPO_URL = "https://github.com/Sushmit-Biswas/thermal-gpu-balancer.git"
REPO_DIR = "/content/clusterops_repo"

def setup_repo():
    # SAFETY: Always move back to /content before deleting to avoid invalid CWD errors
    os.chdir('/content')
    if os.path.exists(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR])
    
    print(f"Cloning repository...")
    result = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"❌ Git clone failed! {result.stderr}")
        return False
        
    os.chdir(REPO_DIR)
    print(f"✅ Successfully moved to {os.getcwd()}")
    return True

if setup_repo():
    # Start server
    server_proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    # Wait for server
    for i in range(15):
        time.sleep(2)
        try:
            if requests.get('http://localhost:8000/health', timeout=2).ok:
                print(f'✅ Server online')
                break
        except: pass

## 4. Load LLM with Unsloth (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch, os

# Set custom cache dir to avoid FileNotFoundError in some Colab environments
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs('/content/hf_cache', exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-1B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)
print('✅ 1B Model loaded')

## 5. GRPO Logic

In [ ]:
import json
ENV_URL = 'http://localhost:8000'
SYSTEM_PROMPT = """Output ONE JSON action only.
  {\"action_type\": \"allocate\", \"job_id\": \"job_1\", \"node_id\": 2}
  {\"action_type\": \"wait\"}"""

def parse_action(text):
    try:
        start, end = text.index('{'), text.rindex('}') + 1
        return json.loads(text[start:end])
    except: return {'action_type': 'wait'}

@torch.no_grad()
def run_llm_episode(model, tokenizer, scenario='01_baseline'):
    # Reset env
    data = requests.post(f'{ENV_URL}/reset', json={'scenario': scenario}).json()
    total_reward = 0.0
    while not data.get('done', False):
        obs, meta = data.get('observation', {}), data.get('metadata', {})
        prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}\n<|eot_id|><|start_header_id|>user<|end_header_id|>\nStep {meta.get('step')}: {obs}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=25, temperature=0.5)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        # Step env
        data = requests.post(f'{ENV_URL}/step', json=parse_action(text)).json()
        total_reward += data.get('reward', 0)
    return total_reward

print('✅ Ready')

## 6. Training

In [ ]:
print("Starting training episodes...")
for ep in range(20):
    reward = run_llm_episode(model, tokenizer, '01_baseline')
    print(f"  Episode {ep+1}: Reward={reward:.1f}")